# Bitcoin Real-Data Inspection

This notebook inspects the contiguous BTC daily log-return series used by the QGAN experiments. The primary stylized-fact estimate uses the full series with moving-block-bootstrap uncertainty. The violin plots use non-overlapping 180-day blocks only to visualize regime variation; they are not treated as independent confidence samples.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CWD = Path.cwd().resolve()
MODEL_ROOT = next((p for base in (CWD, *CWD.parents) for p in (base, base / 'Bitcoin_Data_2020_2026') if p.name == 'Bitcoin_Data_2020_2026' and (p / 'btc_daily_2020_2026.csv').is_file()), None)
if MODEL_ROOT is None:
    raise FileNotFoundError('Cannot locate Bitcoin_Data_2020_2026 from ' + str(CWD))

DATA_FILE = MODEL_ROOT / Path('btc_daily_2020_2026.csv')
FIGURE_DIR = MODEL_ROOT / 'real_data_inspection_figures_2020_2026'
FIGURE_DIR.mkdir(exist_ok=True)
MAX_LAG = 7
BOOTSTRAP_BLOCK_LENGTH = 30
BOOTSTRAP_DRAWS = 2000
REGIME_BLOCK_LENGTH = 180
WINDOW_LENGTH = 16
WINDOW_STRIDE = 3

if not DATA_FILE.is_file():
    raise FileNotFoundError(f'Missing dataset: {DATA_FILE.resolve()}')

data = pd.read_csv(DATA_FILE, parse_dates=['Timestamp'])
returns = data['Log_return'].to_numpy(dtype=float)
data_period = f"{data['Timestamp'].iloc[0].date()} to {data['Timestamp'].iloc[-1].date()}"

assert data['Timestamp'].is_monotonic_increasing
assert not data['Timestamp'].duplicated().any()
assert np.isfinite(returns).all()
print(f'Data period: {data_period}')
print(f'Number of daily returns: {len(returns)}')


In [ ]:
def pearson_correlation(left, right):
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    left_centered = left - left.mean()
    right_centered = right - right.mean()
    denominator = np.sqrt(np.dot(left_centered, left_centered) * np.dot(right_centered, right_centered))
    return np.nan if denominator == 0 else np.dot(left_centered, right_centered) / denominator


def compute_global_metrics(series, max_lag=MAX_LAG):
    series = np.asarray(series, dtype=float).reshape(-1)
    absolute_returns = np.abs(series)
    squared_returns = series**2
    rows = []
    for lag in range(1, max_lag + 1):
        rows.append({
            'lag': lag,
            'raw_acf': pearson_correlation(series[:-lag], series[lag:]),
            'absolute_acf': pearson_correlation(absolute_returns[:-lag], absolute_returns[lag:]),
            'leverage': pearson_correlation(series[:-lag], squared_returns[lag:]),
        })
    return pd.DataFrame(rows)


def moving_block_bootstrap(series, max_lag=MAX_LAG, block_length=BOOTSTRAP_BLOCK_LENGTH,
                           draws=BOOTSTRAP_DRAWS, seed=42):
    """Return full-series metrics and percentile moving-block-bootstrap intervals.

    REVIEW: The 30-day block length preserves the seven displayed lags and
    nearby volatility dependence. Before reporting final intervals, rerun
    with 20, 40, and 60 days to assess block-length sensitivity.
    """
    series = np.asarray(series, dtype=float).reshape(-1)
    if not 1 <= block_length <= len(series):
        raise ValueError('block_length must be between 1 and the series length.')

    point_estimate = compute_global_metrics(series, max_lag=max_lag)
    metric_columns = ['raw_acf', 'absolute_acf', 'leverage']
    bootstrap_values = np.empty((draws, max_lag, len(metric_columns)))
    rng = np.random.default_rng(seed)
    blocks_per_draw = int(np.ceil(len(series) / block_length))
    last_start = len(series) - block_length + 1

    for draw in range(draws):
        # Resampling block starts with replacement preserves order within blocks.
        starts = rng.integers(0, last_start, size=blocks_per_draw)
        resampled = np.concatenate([series[start:start + block_length] for start in starts])[:len(series)]
        bootstrap_values[draw] = compute_global_metrics(resampled, max_lag=max_lag)[metric_columns].to_numpy()

    intervals = {'lag': point_estimate['lag'].to_numpy()}
    for column_index, column in enumerate(metric_columns):
        intervals[f'{column}_lower'] = np.quantile(bootstrap_values[:, :, column_index], 0.025, axis=0)
        intervals[f'{column}_upper'] = np.quantile(bootstrap_values[:, :, column_index], 0.975, axis=0)
    return point_estimate, pd.DataFrame(intervals)


def make_return_windows(series, window_length=WINDOW_LENGTH, stride=WINDOW_STRIDE):
    series = np.asarray(series, dtype=float).reshape(-1)
    if window_length <= 1 or window_length > len(series):
        raise ValueError('window_length must be between 2 and the series length.')
    if stride < 1:
        raise ValueError('stride must be positive.')
    starts = range(0, len(series) - window_length + 1, stride)
    return np.stack([series[start:start + window_length] for start in starts])


def compute_pooled_window_metrics(windows, max_lag=MAX_LAG):
    """Pool valid within-window pairs without creating cross-window pairs."""
    windows = np.asarray(windows, dtype=float)
    if windows.ndim != 2 or windows.shape[1] <= max_lag:
        raise ValueError('windows must be 2D and longer than max_lag.')
    rows = []
    for lag in range(1, max_lag + 1):
        left = windows[:, :-lag].reshape(-1)
        right = windows[:, lag:].reshape(-1)
        rows.append({
            'lag': lag,
            'raw_acf': pearson_correlation(left, right),
            'absolute_acf': pearson_correlation(np.abs(left), np.abs(right)),
            'leverage': pearson_correlation(left, right**2),
        })
    return pd.DataFrame(rows)


def compute_individual_window_metrics(windows, max_lag=MAX_LAG):
    """Return the distribution of bias-corrected short-window estimators."""
    windows = np.asarray(windows, dtype=float)
    if windows.ndim != 2 or windows.shape[1] <= max_lag:
        raise ValueError('windows must be 2D and longer than max_lag.')

    # Global centering avoids fitting a separate noisy mean to every 16-return window.
    raw_mean = windows.mean()
    absolute_mean = np.abs(windows).mean()
    squared_mean = np.square(windows).mean()
    window_length = windows.shape[1]
    rows = []

    for window_index, window in enumerate(windows):
        raw = window - raw_mean
        absolute = np.abs(window) - absolute_mean
        squared = np.square(window) - squared_mean
        raw_norm = np.dot(raw, raw)
        absolute_norm = np.dot(absolute, absolute)
        squared_norm = np.dot(squared, squared)

        for lag in range(1, max_lag + 1):
            correction = window_length / (window_length - lag)
            rows.append({
                'window': window_index,
                'lag': lag,
                'raw_acf': np.nan if raw_norm == 0 else correction * np.dot(raw[:-lag], raw[lag:]) / raw_norm,
                'absolute_acf': np.nan if absolute_norm == 0 else correction * np.dot(absolute[:-lag], absolute[lag:]) / absolute_norm,
                'leverage': np.nan if raw_norm == 0 or squared_norm == 0 else correction * np.dot(raw[:-lag], squared[lag:]) / np.sqrt(raw_norm * squared_norm),
            })

    return pd.DataFrame(rows)


def moving_block_bootstrap_window_metrics(series, max_lag=MAX_LAG, block_length=BOOTSTRAP_BLOCK_LENGTH,
                                         draws=BOOTSTRAP_DRAWS, seed=42):
    """Bootstrap the contiguous path, then rebuild matched 16-return windows."""
    series = np.asarray(series, dtype=float).reshape(-1)
    windows = make_return_windows(series)
    point_estimate = compute_pooled_window_metrics(windows, max_lag=max_lag)
    metric_columns = ['raw_acf', 'absolute_acf', 'leverage']
    bootstrap_values = np.empty((draws, max_lag, len(metric_columns)))
    rng = np.random.default_rng(seed)
    blocks_per_draw = int(np.ceil(len(series) / block_length))
    last_start = len(series) - block_length + 1

    for draw in range(draws):
        starts = rng.integers(0, last_start, size=blocks_per_draw)
        resampled = np.concatenate([series[start:start + block_length] for start in starts])[:len(series)]
        resampled_windows = make_return_windows(resampled)
        bootstrap_values[draw] = compute_pooled_window_metrics(
            resampled_windows, max_lag=max_lag
        )[metric_columns].to_numpy()

    intervals = {'lag': point_estimate['lag'].to_numpy()}
    for column_index, column in enumerate(metric_columns):
        intervals[f'{column}_lower'] = np.quantile(bootstrap_values[:, :, column_index], 0.025, axis=0)
        intervals[f'{column}_upper'] = np.quantile(bootstrap_values[:, :, column_index], 0.975, axis=0)
    return point_estimate, pd.DataFrame(intervals), len(windows)


def compute_nonoverlapping_block_metrics(series, timestamps, block_length=REGIME_BLOCK_LENGTH, max_lag=MAX_LAG):
    rows = []
    for start in range(0, len(series) - block_length + 1, block_length):
        metrics = compute_global_metrics(series[start:start + block_length], max_lag=max_lag)
        metrics['block_start'] = timestamps.iloc[start]
        metrics['block_end'] = timestamps.iloc[start + block_length - 1]
        rows.append(metrics)
    return pd.concat(rows, ignore_index=True)


## Distribution and Tail Diagnostics

The histogram, ECDF, and normal QQ plot describe the marginal return distribution. The logarithmic count histogram is useful because rare tail observations are otherwise visually hidden.

In [ ]:
summary = pd.Series(returns).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_frame('Log_return')
summary.loc['skewness'] = stats.skew(returns, bias=False)
summary.loc['excess_kurtosis'] = stats.kurtosis(returns, fisher=True, bias=False)
summary.round(6)


In [ ]:
sorted_returns = np.sort(returns)
ecdf = np.arange(1, len(sorted_returns) + 1) / len(sorted_returns)
normal_quantiles = stats.norm.ppf((np.arange(1, len(returns) + 1) - 0.5) / len(returns))
sample_quantiles = np.sort((returns - returns.mean()) / returns.std(ddof=1))
qq_limit = max(np.abs(normal_quantiles).max(), np.abs(sample_quantiles).max())

distribution_figure = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Histogram with density normalization', 'Histogram with logarithmic counts',
                    'Empirical cumulative distribution', 'Normal QQ plot of standardized returns')
)
distribution_figure.add_trace(go.Histogram(x=returns, histnorm='probability density', nbinsx=80, marker_color='#1f77b4', showlegend=False), row=1, col=1)
distribution_figure.add_trace(go.Histogram(x=returns, nbinsx=80, marker_color='#1f77b4', showlegend=False), row=1, col=2)
distribution_figure.add_trace(go.Scatter(x=sorted_returns, y=ecdf, mode='lines', line=dict(color='#1f77b4'), showlegend=False), row=2, col=1)
distribution_figure.add_trace(go.Scatter(x=normal_quantiles, y=sample_quantiles, mode='markers', marker=dict(size=4, color='#1f77b4', opacity=0.6), showlegend=False), row=2, col=2)
distribution_figure.add_trace(go.Scatter(x=[-qq_limit, qq_limit], y=[-qq_limit, qq_limit], mode='lines', line=dict(color='black', dash='dash'), showlegend=False), row=2, col=2)
distribution_figure.update_yaxes(type='log', row=1, col=2)
distribution_figure.update_xaxes(title_text='Daily log return', row=1, col=1)
distribution_figure.update_xaxes(title_text='Daily log return', row=1, col=2)
distribution_figure.update_xaxes(title_text='Daily log return', row=2, col=1)
distribution_figure.update_xaxes(title_text='Theoretical normal quantile', row=2, col=2)
distribution_figure.update_yaxes(title_text='Density', row=1, col=1)
distribution_figure.update_yaxes(title_text='Count (log scale)', row=1, col=2)
distribution_figure.update_yaxes(title_text='ECDF', row=2, col=1)
distribution_figure.update_yaxes(title_text='Sample standardized quantile', row=2, col=2)
distribution_figure.update_layout(title=f'BTC Daily Log-Return Distribution ({data_period})', template='plotly_white', height=800)
distribution_figure.write_html(FIGURE_DIR / 'distribution_diagnostics.html', include_plotlyjs=True)
distribution_figure


## Primary Stylized-Fact Benchmark

This is the primary empirical reference for later QGAN evaluation: one full-series estimate at each lag, with moving-block-bootstrap confidence intervals.

In [ ]:
global_metrics, global_intervals = moving_block_bootstrap(returns)
global_metrics.round(4)


In [ ]:
metric_definitions = [
    ('raw_acf', 'Raw Returns Autocorrelation', 'ACF', '#1f77b4'),
    ('absolute_acf', 'Absolute Returns Autocorrelation', 'ACF', '#ff7f0e'),
    ('leverage', 'Leverage Effect: Return vs Future Squared Return', 'Correlation', '#2ca02c'),
]

global_figure = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                             subplot_titles=[item[1] for item in metric_definitions])
for row, (column, _, ylabel, color) in enumerate(metric_definitions, start=1):
    lower = global_intervals[f'{column}_lower']
    upper = global_intervals[f'{column}_upper']
    global_figure.add_trace(go.Scatter(x=global_metrics['lag'], y=upper, mode='lines', line=dict(width=0), hoverinfo='skip', showlegend=False), row=row, col=1)
    global_figure.add_trace(go.Scatter(x=global_metrics['lag'], y=lower, mode='lines', line=dict(width=0), fill='tonexty', fillcolor='rgba(31, 119, 180, 0.20)', name='95% moving-block bootstrap CI', legendgroup='ci', showlegend=(row == 1)), row=row, col=1)
    global_figure.add_trace(go.Scatter(x=global_metrics['lag'], y=global_metrics[column], mode='lines+markers', line=dict(color=color, width=3), name='Full-series estimate', legendgroup='estimate', showlegend=(row == 1)), row=row, col=1)
    global_figure.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.6, row=row, col=1)
    global_figure.update_yaxes(title_text=ylabel, row=row, col=1)
global_figure.update_xaxes(title_text='Lag (days)', row=3, col=1, dtick=1)
global_figure.update_layout(title=f'Global BTC Stylized Facts with Bootstrap Uncertainty ({data_period})', template='plotly_white', height=900)
global_figure.write_html(FIGURE_DIR / 'global_stylized_metrics.html', include_plotlyjs=True)
global_metrics.to_csv(FIGURE_DIR / 'global_stylized_metrics.csv', index=False)
global_intervals.to_csv(FIGURE_DIR / 'global_stylized_metric_intervals.csv', index=False)
global_figure


## Matched 16-Return-Window Benchmark

The generator produces independent 16-return trajectories. This matched empirical estimate therefore pools only valid pairs within 16-return windows extracted at the training stride of 3; it never joins two windows. Its moving-block-bootstrap intervals resample the original contiguous BTC path and rebuild the windows, so overlapping real windows are not incorrectly treated as independent observations.

In [ ]:
window_metrics, window_intervals, n_windows = moving_block_bootstrap_window_metrics(returns)
print(f'Matched empirical windows: {n_windows} windows of {WINDOW_LENGTH} returns at stride {WINDOW_STRIDE}')
window_metrics.round(4)


In [ ]:
window_figure = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                                  subplot_titles=[item[1] for item in metric_definitions])
for row, (column, _, ylabel, color) in enumerate(metric_definitions, start=1):
    lower = window_intervals[f'{column}_lower']
    upper = window_intervals[f'{column}_upper']
    window_figure.add_trace(go.Scatter(x=window_metrics['lag'], y=upper, mode='lines', line=dict(width=0), hoverinfo='skip', showlegend=False), row=row, col=1)
    window_figure.add_trace(go.Scatter(x=window_metrics['lag'], y=lower, mode='lines', line=dict(width=0), fill='tonexty', fillcolor='rgba(214, 39, 40, 0.20)', name='95% moving-block bootstrap CI', legendgroup='window_ci', showlegend=(row == 1)), row=row, col=1)
    window_figure.add_trace(go.Scatter(x=window_metrics['lag'], y=window_metrics[column], mode='lines+markers', line=dict(color='#d62728', width=3), name='Pooled 16-return-window estimate', legendgroup='window_estimate', showlegend=(row == 1)), row=row, col=1)
    window_figure.add_trace(go.Scatter(x=global_metrics['lag'], y=global_metrics[column], mode='lines+markers', line=dict(color=color, dash='dot', width=2), name='Full contiguous-series estimate', legendgroup='global_estimate', showlegend=(row == 1)), row=row, col=1)
    window_figure.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.6, row=row, col=1)
    window_figure.update_yaxes(title_text=ylabel, row=row, col=1)
window_figure.update_xaxes(title_text='Lag (days)', row=3, col=1, dtick=1)
window_figure.update_layout(title=f'Matched 16-Return-Window BTC Stylized Facts ({data_period})', template='plotly_white', height=900)
window_figure.write_html(FIGURE_DIR / 'matched_window_stylized_metrics.html', include_plotlyjs=True)
window_metrics.to_csv(FIGURE_DIR / 'matched_window_stylized_metrics.csv', index=False)
window_intervals.to_csv(FIGURE_DIR / 'matched_window_stylized_metric_intervals.csv', index=False)
window_figure


## Matched 16-Return-Window Violin Diagnostic

These violins show the distribution of individual 16-return-window estimators computed from the same stride-3 real-data windows used for training. They are deliberately a matched visual reference for generated 16-return trajectories, rather than confidence intervals or primary estimates of the population stylized facts. Their broad spread is expected because a lag-7 correlation within a 16-return window uses only nine paired observations.

In [ ]:
matched_windows = make_return_windows(returns)
matched_window_violin_metrics = compute_individual_window_metrics(matched_windows)
print(f'Violin diagnostic: {len(matched_windows)} matched windows of {WINDOW_LENGTH} returns at stride {WINDOW_STRIDE}')

short_window_violin_figure = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=[item[1] + ' Across Individual 16-Return Windows' for item in metric_definitions],
)
for row, (column, _, ylabel, color) in enumerate(metric_definitions, start=1):
    for lag in global_metrics['lag']:
        subset = matched_window_violin_metrics.loc[
            matched_window_violin_metrics['lag'] == lag, column
        ].dropna()
        short_window_violin_figure.add_trace(go.Violin(
            x=np.repeat(f'Lag {lag}', len(subset)), y=subset, name=f'Lag {lag}',
            box_visible=True, meanline_visible=True, points=False,
            line=dict(color=color), fillcolor=color, opacity=0.72,
            showlegend=False, scalemode='width',
        ), row=row, col=1)
    short_window_violin_figure.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.6, row=row, col=1)
    short_window_violin_figure.update_yaxes(title_text=ylabel, row=row, col=1)
short_window_violin_figure.update_xaxes(title_text='Lag (days)', row=3, col=1)
short_window_violin_figure.update_layout(
    title=f'Distribution of 16-Return-Window Stylized-Fact Estimators ({data_period}; stride {WINDOW_STRIDE})',
    template='plotly_white', height=1000,
)
short_window_violin_figure.write_html(FIGURE_DIR / 'matched_window_metric_violins.html', include_plotlyjs=True)
matched_window_violin_metrics.to_csv(FIGURE_DIR / 'matched_window_metric_violins.csv', index=False)
short_window_violin_figure


## Regime Variation Diagnostic

The 16-return-window violins above are retained only as a matched short-trajectory diagnostic. The following violin plots use non-overlapping 180-day blocks, so each correlation uses roughly 173--179 pairs. They visualize how the stylized facts vary across market regimes, not estimator uncertainty.

In [ ]:
block_metrics = compute_nonoverlapping_block_metrics(returns, data['Timestamp'])
block_metrics['block_label'] = block_metrics['block_start'].dt.strftime('%Y-%m-%d') + ' to ' + block_metrics['block_end'].dt.strftime('%Y-%m-%d')
print(f'Number of non-overlapping {REGIME_BLOCK_LENGTH}-day blocks: {block_metrics["block_label"].nunique()}')

violin_figure = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                             subplot_titles=[item[1] + ' Across Non-Overlapping 180-Day Blocks' for item in metric_definitions])
for row, (column, _, ylabel, color) in enumerate(metric_definitions, start=1):
    for lag in global_metrics['lag']:
        subset = block_metrics.loc[block_metrics['lag'] == lag, column]
        violin_figure.add_trace(go.Violin(
            x=np.repeat(f'Lag {lag}', len(subset)), y=subset, name=f'Lag {lag}',
            box_visible=True, meanline_visible=True, points='all', jitter=0.08,
            marker=dict(size=5, opacity=0.55, color=color), line=dict(color=color),
            showlegend=False, scalemode='width'
        ), row=row, col=1)
    violin_figure.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.6, row=row, col=1)
    violin_figure.update_yaxes(title_text=ylabel, row=row, col=1)
violin_figure.update_xaxes(title_text='Lag (days)', row=3, col=1)
violin_figure.update_layout(title='Regime Variation of BTC Stylized Facts: Non-Overlapping 180-Day Blocks', template='plotly_white', height=1000)
violin_figure.write_html(FIGURE_DIR / 'block_metric_violins.html', include_plotlyjs=True)
violin_figure


In [ ]:
lag_one_blocks = block_metrics.loc[block_metrics['lag'] == 1].copy()
rolling_figure = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                              subplot_titles=['Lag-1 Raw ACF by 180-Day Block', 'Lag-1 Absolute ACF by 180-Day Block', 'Lag-1 Leverage by 180-Day Block'])
for row, (column, _, ylabel, color) in enumerate(metric_definitions, start=1):
    rolling_figure.add_trace(go.Scatter(x=lag_one_blocks['block_end'], y=lag_one_blocks[column], mode='lines+markers', line=dict(color=color, width=2), marker=dict(size=7), showlegend=False), row=row, col=1)
    rolling_figure.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.6, row=row, col=1)
    rolling_figure.update_yaxes(title_text=ylabel, row=row, col=1)
rolling_figure.update_xaxes(title_text='Block end date', row=3, col=1)
rolling_figure.update_layout(title='Lag-1 Stylized Facts Across Bitcoin Market Regimes', template='plotly_white', height=850)
rolling_figure.write_html(FIGURE_DIR / 'lag_one_regime_variation.html', include_plotlyjs=True)
rolling_figure


## Interpretation Checklist

- Use the global curves and their bootstrap intervals for empirical claims.
- Use the matched 16-return-window violin only to compare short-trajectory estimator variability with QGAN outputs.
- Use the 180-day block violin and time plots to discuss time variation, not confidence intervals.
- Compare a QGAN only after generated returns are returned to the original log-return scale.
- The block-bootstrap confidence intervals should be checked for sensitivity to block lengths of 20, 40, and 60 days before paper submission.